In [61]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from datetime import datetime

# 自作モジュールの読み込み
sys.path.append(os.path.abspath('..'))
from configs.config import *
from src.runner import Runner
from src.model_LGBM import model_LGBM
from src.util import Logger, Util

In [62]:
# ロガーの設定
logger = Logger(path=DIR_LOG)

def get_run_name(model_type):
    """run名の作成
    """
    run_name = model_type
    suffix = '_' + datetime.now().strftime("%Y%m%d%H%M")
    run_name = run_name + suffix
    return run_name

# LGBM_base

In [63]:
# Key
df_all = Util.load_feature('Key')
# 目的変数
key_col = ['社員番号', 'category']
df_all = pd.merge(df_all, Util.load_feature('Target'), on=key_col, how='left')\
# category特徴量
key_col = 'category'
list_feaqture_name = [
    'CategoryFeature',
]
for feature_name in list_feaqture_name:
    df_feature = Util.load_feature(feature_name)
    df_all = pd.merge(df_all, df_feature, on=key_col, how='left')
# 社員特徴量
key_col = '社員番号'
list_feaqture_name = [
    'CareerFeature',
    'DxFeature',
    'HrFeature',
    'OvertimeWorkByMonthFeature',
    'PositionHistoryFeature',
    'UdemyActivityFeature',
]
for feature_name in list_feaqture_name:
    df_feature = Util.load_feature(feature_name)
    df_all = pd.merge(df_all, df_feature, on=key_col, how='left')

# train test
df_train = df_all[df_all['target'].notnull()]
df_test = df_all[df_all['target'].isnull()]

In [64]:
# run_nameの設定
run_name = get_run_name(model_type="lgbm_base")
# run_name = 'lgbm_multimodel_bikes_202410261847'
run_name

'lgbm_base_202507281705'

In [65]:
pos = sum(df_train['target'] == 1)
neg = sum(df_test['target'] != 1)

print(neg / pos)

38.13840830449827


In [66]:
def after_predict_process(df_pred, target_col):
    """予測後に行う処理
    Args:
        df_pred(pd.DataFrame): 予測データ[key_cols, 予測値]
        target_col(str): 予測値のカラム名
    Returns:    
        df_pred(pd.DataFrame): 予測データ[key_cols, 予測値]
    """
    return df_pred

def after_split_process(tr, va):
    """データセットの分割後に行う処理
    Args:
        tr(pd.DataFrame): 訓練データ
        va(pd.DataFrame): 検証データ
    returns:
        tr(pd.DataFrame): 訓練データ
        va(pd.DataFrame): 検証データ
    """
    return tr, va

model_params_lgb = {
    #### run params
    "key_cols": KEY_COL,                    # ユニークキー
    "target_col": TARGET_COL,              # 目的変数（0 or 1）
    "remove_cols": [],
    #### model train params
    "num_boost_round": 5000,
    "early_stopping_rounds": 100,
    "verbose": -1,
    "period": 100,
    "log_level": 'error',
    "verbosity": -1,
    #### model core params (for binary classification)
    "boosting_type": "gbdt",
    "objective": "binary",                 
    "metric": "auc",                       
    "learning_rate": 0.001,
    "scale_pos_weight": 19,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,              # 追加: 通常は併用
    "bagging_freq": 1,                    # 追加: バギングの頻度
    "num_leaves": 31,                     # 追加: 複雑さの制御
    "min_data_in_leaf": 20,               # 追加: 過学習抑制
    "max_depth": -1,                      # 追加: 自由に展開
    "random_state": 42,                   # 追加: 再現性確保
}
run_setting = {
    'calc_shap': False,     # shap値を計算するか否か
    "tune_params": True,           # パラメータチューニング、lgb_hopt,xgb_hopt,nn_hopt,False
    "after_predict_process": after_predict_process,
    'after_split_process': after_split_process,  
}
cv_setting = {
    "target_col": TARGET_COL,
    "group_col": "社員番号",  # グループ化するカラム
    "n_splits": 5,  # 分割数,
    "shuffle": True,  # シャッフルするか否か
    "random_state": 42,  # ランダムシード
}

In [67]:
import importlib
from src import runner
from src import model_LGBM
from src import model
from configs import config

# runnerモジュールをリロード
importlib.reload(runner)
importlib.reload(model_LGBM)
importlib.reload(model)
importlib.reload(config)

# Runnerクラスを再インポート
from src.model import Model
from src.runner import Runner
from src.model_LGBM import model_LGBM
from configs.config import *

In [68]:
memo = "LightGBM scale_pos_weight==38"
ml_runner = Runner(
    run_name,
    model_LGBM,
    model_params_lgb,
    df_train,
    df_test,
    run_setting,
    cv_setting,
    logger,
    memo,
)

In [69]:
# ml_runner.tune_params(30)

In [70]:
ml_runner.run_train_cv()

[2025-07-28 17:05:44] - lgbm_base_202507281705 - start training cv
[2025-07-28 17:05:44] - lgbm_base_202507281705 fold 0 - start training


Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.913185	eval's auc: 0.695314
[200]	train's auc: 0.925222	eval's auc: 0.701396
Early stopping, best iteration is:
[176]	train's auc: 0.922526	eval's auc: 0.702969


[2025-07-28 17:05:46] - lgbm_base_202507281705 fold 0 - end training
[2025-07-28 17:05:46] - lgbm_base_202507281705 fold 1 - start training


Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.907393	eval's auc: 0.716516
[200]	train's auc: 0.920849	eval's auc: 0.723392
[300]	train's auc: 0.930336	eval's auc: 0.720855
Early stopping, best iteration is:
[201]	train's auc: 0.921082	eval's auc: 0.723649


[2025-07-28 17:05:49] - lgbm_base_202507281705 fold 1 - end training
[2025-07-28 17:05:49] - lgbm_base_202507281705 fold 2 - start training


...

[2025-07-28 17:05:52] - lgbm_base_202507281705 fold 2 - end training
[2025-07-28 17:05:52] - lgbm_base_202507281705 fold 3 - start training


Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.917799	eval's auc: 0.645785
Early stopping, best iteration is:
[20]	train's auc: 0.887951	eval's auc: 0.648562


[2025-07-28 17:05:54] - lgbm_base_202507281705 fold 3 - end training
[2025-07-28 17:05:54] - lgbm_base_202507281705 fold 4 - start training


Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.904575	eval's auc: 0.648686
Early stopping, best iteration is:
[16]	train's auc: 0.872953	eval's auc: 0.656238


[2025-07-28 17:05:55] - lgbm_base_202507281705 fold 4 - end training
[2025-07-28 17:05:55] - lgbm_base_202507281705 - end training cv


In [71]:
ml_runner.run_metric_cv()

[2025-07-28 17:05:55] - lgbm_base_202507281705 - start metric cv
100%|██████████| 5/5 [00:01<00:00,  2.90it/s]
memo: LightGBM scale_pos_weight==38
run_name:lgbm_base_202507281705	score_mean:0.6626156700251687	score0:0.7029685906734262	score1:0.7236492351090377	score2:0.581660756501182	score3:0.6485621597844615	score4:0.6562376080577366
mean: 0.6626156700251687, std: 0.0492855620236062
[2025-07-28 17:05:57] - mean: 0.6626156700251687, std: 0.0492855620236062
[2025-07-28 17:05:57] - output predict : g:\マイドライブ\competitions\atma_udemy\models\lgbm_base_202507281705\va_pred.pkl
[2025-07-28 17:05:57] - lgbm_base_202507281705 - end metric cv


In [72]:
ml_runner.run_predict_cv()

[2025-07-28 17:05:57] - lgbm_base_202507281705 - start prediction cv
100%|██████████| 5/5 [00:00<00:00, 17.99it/s]
[2025-07-28 17:05:57] - output predict : g:\マイドライブ\competitions\atma_udemy\models\lgbm_base_202507281705\te_pred.pkl
[2025-07-28 17:05:57] - lgbm_base_202507281705 - end prediction cv


In [73]:
ml_runner.plot_feature_importance_cv()

[2025-07-28 17:05:57] - lgbm_base_202507281705 - start plot feature importance cv
[2025-07-28 17:06:08] - lgbm_base_202507281705 - end plot feature importance cv


# Submissionの作成

In [74]:
runner = ml_runner

In [75]:
df_te_pred = pd.read_pickle(os.path.join(runner.out_dir_name, "te_pred.pkl"))
df_prep_test = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_test.pkl"))
df_pred = pd.merge(df_prep_test, df_te_pred, on=["社員番号", "category"], how="left")
df_submit = df_pred[['target']]

In [76]:
path_submit = os.path.join(DIR_SUBMISSIONS, f"{runner.run_name}_submition.csv")
df_submit.to_csv(path_submit, header=True, index=False)
print(path_submit)
pd.read_csv(path_submit)

g:\マイドライブ\competitions\atma_udemy\data\submission\lgbm_base_202507281705_submition.csv


,target
0,0.057403
1,0.057238
2,0.063992
3,0.057492
4,0.057566
...,...
11017,0.089640
11018,0.089802
11019,0.089728
11020,0.091984


In [77]:
df_submit['target'].describe()

count    11022.000000
mean         0.069722
std          0.018872
min          0.038401
25%          0.054185
50%          0.067323
75%          0.083244
max          0.118886
Name: target, dtype: float64